# Self-Attention from Scratch

Attention is a lookup table where every word asks "who matters to me?" - and learns the answer.

## Problem definition

RNNs process sequences one token at a time. By the time you reach token 50, the information from token 1 has been squeezed through 50 compression steps. Long-range dependencies get crushed into a fixed-size hidden state - a bottleneck that no amount of LSTM gating fully solves.

Self-attention lets every position in a sequence attend to every other position in a single parallel step. That is what makes transformers fast, scalable and dominant.

## Basic Concept

### The Database Lookup Analogy

Attention is a soft database lookup:
    Sum of weights * value

Every token generate three vectors:
* Query (Q) : "What am i looking for"
* Key (K) : "What do i contain"
* Value (V): "What information do i provide if selected"

The dot product between a query and all keys produces attention scores, high score means "this key matches my query". Those scores weight the value.

### Q, K, V Computation

Each token embedding get projected through **three learned weight matrix**: 

```
Input Embedding (Sequence of n tokens, each d-dimensional):

X = [x1, x2, x3, ..., xn]   Shape: (n, d)

Three weight matrices:

Wq shape: (d, dk)
Wk shape: (d, dk)
Wv shape: (d, dv)

Projection:
Q = X @ Wq    shape: (n, dk)
K = X @ Wk    shape: (n, dk)
V = X @ Wv    shape: (n, dv)
```

### The attention matrix

```
Scores = Q @ K^T shape: (n, n)
```
||k1|k2|k3|
|---|---|---|---|
|q1|2.1|0.3|0.1|
|q2|0.4|1.9|0.7|
|q3|0.2|0.6|2.3|

Each row: one token's attention over entire sequence.

### Scale

The dot products grow with dimension $d_k$. Fix: divide by $\sqrt{d_k}$.

**Why dot products scale with $d_k$**

A single score is $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ — a sum of $d_k$ terms.

Assume each component is independent with zero mean and unit variance:

$$\mathbb{E}[q_i] = \mathbb{E}[k_i] = 0, \quad \mathrm{Var}(q_i) = \mathrm{Var}(k_i) = 1$$

Then the dot product has zero mean but variance that grows with $d_k$:

$$\mathrm{Var}(q \cdot k) = \mathrm{Var}\!\left(\sum_{i=1}^{d_k} q_i k_i\right) = \sum_{i=1}^{d_k} \mathrm{Var}(q_i k_i) = d_k$$

Each term $\mathrm{Var}(q_i k_i) = 1$ because $q_i, k_i$ are independent with unit variance. So $\mathrm{Std}(q \cdot k) = \sqrt{d_k}$ — at $d_k = 64$, scores often land in the tens.

**Fix: scale by $\sqrt{d_k}$**

$$\mathrm{Var}\!\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = \frac{1}{d_k}\,\mathrm{Var}(q \cdot k) = 1$$

Variance stays at 1 regardless of $d_k$, keeping softmax inputs in a stable range.

```
Scaled Scores = (Q @ K^T) / sqrt(dk)
```

### Softmax Turns Scores into Weights



# Build your Own

## Softmax from Scratch

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))

from user_tools import SectionPrinter

import numpy as np

def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=-1, keepdims=True)

with SectionPrinter("Softmax from Scratch"):
    logits = np.array([
        [1, 2, 3],
        [1, 3, 5],
        [1, 4, 7]
    ])
    print(softmax(logits))


====================Softmax from Scratch====================
[[0.09003057 0.24472847 0.66524096]
 [0.01587624 0.11731043 0.86681333]
 [0.00235563 0.04731416 0.95033021]]


## Scaled dot-product attention

In [12]:
def scaled_dot_product_attention(Q, K, V):
    """NumPy version for 2D inputs: Q, K, V shape (seq, d)."""
    dk = Q.shape[-1]
    scores = Q @ K.T / (dk ** 0.5)
    weights = softmax(scores)
    output = weights @ V
    return output, weights

with SectionPrinter("Scaled dot-product attention"):
    Q = np.random.randn(3, 3)
    K = np.random.randn(3, 3)
    V = np.random.randn(3, 3)
    output, weights = scaled_dot_product_attention(Q, K, V)
    print(output)
    print(weights)

================Scaled dot-product attention================
[[ 0.62406962  0.09154135 -1.41658723]
 [-0.55208827  1.38605023 -0.74461513]
 [ 0.21856298  0.70561416 -0.95609989]]
[[0.20039227 0.4788829  0.32072483]
 [0.01716783 0.12501219 0.85781998]
 [0.23522561 0.23294554 0.53182885]]


## Self-attention class with learned projections

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def torch_scaled_dot_product_attention(Q, K, V):
    dk = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (dk ** 0.5)
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights


class SelfAttention(nn.Module):
    def __init__(self, d_model, dk, dv):
        super().__init__()
        self.Wq = nn.Linear(d_model, dk, bias=False)
        self.Wk = nn.Linear(d_model, dk, bias=False)
        self.Wv = nn.Linear(d_model, dv, bias=False)

    def forward(self, X):
        Q = self.Wq(X)
        K = self.Wk(X)
        V = self.Wv(X)
        return torch_scaled_dot_product_attention(Q, K, V)


with SectionPrinter("Self-attention"):
    d_model = 4
    dk = 2
    dv = 3
    X = torch.randn(2, 3, d_model)  # (batch, seq, d_model)
    attention = SelfAttention(d_model, dk, dv)
    outputs, weights = attention(X)
    print(f"outputs: {outputs.shape}, weights: {weights.shape}")

=======================Self-attention=======================
outputs: torch.Size([2, 3, 3]), weights: torch.Size([2, 3, 3])


## Pytorch's Implement

In [19]:
mha = nn.MultiheadAttention(
    embed_dim=6,
    num_heads=2,
    batch_first=True
)

# Batch size 1, sequence length 4, embedding dimension 6
X_torch = torch.randn(1, 4, 6)

output, attn_weights = mha(X_torch, X_torch, X_torch)

print(output.shape)
print(attn_weights.shape)


torch.Size([1, 4, 6])
torch.Size([1, 4, 4])


## `nn.MultiheadAttention` 源码导读

源码位置：`torch/nn/modules/activation.py` → `forward()` 最终调用 `F.multi_head_attention_forward()`。

### `__init__`：三个设计要点

| 参数 | 源码行为 | 含义 |
|------|----------|------|
| `embed_dim`, `num_heads` | `head_dim = embed_dim // num_heads` | 每个 head 的维度 \(d_k = d_v\) |
| `in_proj_weight` | shape `(3 * embed_dim, embed_dim)` | **Q/K/V 投影合并成一个大矩阵**，一次 `linear` 算完 |
| `out_proj` | `Linear(embed_dim, embed_dim)` | 多头 concat 后再投影回 `embed_dim` |

`in_proj_weight` 按行切成三块：`W_q, W_k, W_v`，各 `(embed_dim, embed_dim)`。

### `forward()` 执行路径

```
输入 query/key/value
    │
    ├─ [fast path] 条件满足 → torch._native_multi_head_attention（融合 CUDA kernel）
    │
    └─ [slow path] F.multi_head_attention_forward
           1. batch_first=True 时先 transpose → (seq, batch, dim)  内部默认格式
           2. _in_projection_packed → 一次算出 Q, K, V
           3. reshape + transpose → (batch*heads, seq, head_dim)
           4. scaled_dot_product_attention(Q, K, V)  ← 和我们手写的一样
           5. reshape + out_proj
           6. batch_first=True 时再 transpose 回 (batch, seq, dim)
```

### 与我们 `SelfAttention` 的对比

| | 我们的实现 | `nn.MultiheadAttention` |
|---|---|---|
| Q/K/V 投影 | 三个独立 `nn.Linear` | 一个 `in_proj_weight` 打包 |
| Head 切分 | 无（单头） | `view` + `transpose` 拆成 `num_heads` |
| Attention 核心 | `Q @ K.T / sqrt(dk)` + softmax | 同样，但走 `scaled_dot_product_attention`（可启用 FlashAttention） |
| 输出投影 | 无 | `out_proj` |
| 默认 layout | `(batch, seq, dim)` | 内部 `(seq, batch, dim)`，需 `batch_first=True` |

### 关键 reshape（多头拆分的核心）

```python
# q: (tgt_len, batch, embed_dim) → (batch*heads, tgt_len, head_dim)
q = q.view(tgt_len, batch * num_heads, head_dim).transpose(0, 1)
```

每个 head 独立做 attention，最后在 `head_dim` 维 concat，再经 `out_proj`。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

embed_dim, num_heads = 6, 2
head_dim = embed_dim // num_heads
batch, seq = 1, 4

mha = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
X = torch.randn(batch, seq, embed_dim)

with SectionPrinter("MHA source walkthrough"):
    # --- __init__: packed QKV weight ---
    W = mha.in_proj_weight  # (3*embed_dim, embed_dim)
    Wq, Wk, Wv = W.chunk(3, dim=0)
    print(f"in_proj_weight: {W.shape}  ->  Wq/Wk/Wv each {Wq.shape}")

    # --- forward step 1: project Q, K, V (same as F.linear) ---
    Q = F.linear(X, Wq, mha.in_proj_bias[:embed_dim])
    K = F.linear(X, Wk, mha.in_proj_bias[embed_dim : 2 * embed_dim])
    V = F.linear(X, Wv, mha.in_proj_bias[2 * embed_dim :])
    print(f"Q/K/V: {Q.shape}  (batch, seq, embed_dim)")

    # --- forward step 2: split heads ---
    Qh = Q.view(batch, seq, num_heads, head_dim).transpose(1, 2)  # (B, H, S, D)
    Kh = K.view(batch, seq, num_heads, head_dim).transpose(1, 2)
    Vh = V.view(batch, seq, num_heads, head_dim).transpose(1, 2)
    print(f"per-head Q: {Qh.shape}  (batch, heads, seq, head_dim)")

    # --- forward step 3: scaled dot-product attention (per head) ---
    scores = Qh @ Kh.transpose(-2, -1) / (head_dim ** 0.5)
    weights = F.softmax(scores, dim=-1)
    attn_out = weights @ Vh  # (B, H, S, D)
    print(f"attention weights: {weights.shape}")

    # --- forward step 4: concat heads + out_proj ---
    merged = attn_out.transpose(1, 2).reshape(batch, seq, embed_dim)
    manual_out = mha.out_proj(merged)

    # --- compare with nn.MultiheadAttention ---
    official_out, official_w = mha(X, X, X, need_weights=True)
    print(f"\nmanual vs official max diff: {(manual_out - official_out).abs().max():.2e}")
    print(f"official output: {official_out.shape}, weights: {official_w.shape}")